# PrecisionSkin / LUMYVUE - Fitzpatrick17k skin-type model (MobileNetV2)

A **separate** model from the four-class skin-condition classifier. It never changes that classifier and is not an output of it.
The two results are only combined later, at the WPF UI / database / patient record / report level.

**How to use:** run the cells **one at a time, top to bottom**, and read the Markdown above each cell first (what it does, what you should see, what to check before going on).
Kernel: the WSL `.venv` of `~/PrecisionSkinV1` (it has TensorFlow, tf2onnx, onnxruntime, scikit-learn, imagehash).

**Safety rules built into this notebook**
- Your dataset folder is **read-only** here: nothing is renamed, moved, merged, relabeled or deleted. Every output goes to `OUTPUT_ROOT`.
- Your folder names are the labels. Type I and Type II stay separate. Nothing is merged. Suspicious images are **flagged in CSV files**, never relabeled.
- The test set is created once, **frozen** (hash-checked) and only evaluated when you switch `RUN_TEST_EVALUATION` on.
- Color is not touched: no hue/saturation/brightness jitter, no CLAHE/histogram equalization/white-balance by default.

> **Scientific limitation (kept in the exported metadata):** Fitzpatrick Skin Type is **not simply a visual skin-colour category** (it is defined by how skin reacts to sun exposure).
> This model outputs an **AI-predicted Fitzpatrick category based on the dataset's labels**, *not* a clinical determination of a person's actual phototype.
> Fitzpatrick17k-style images are mostly skin-condition photographs from many body sites; webcam face images are a different domain, so performance on the Logitech BRIO captures must be checked separately.

## 00 - Environment & Configuration
**What it does:** one place for every setting, seeds everything, imports TensorFlow, prints versions and whether a GPU is visible, creates the output folders.
**Expect:** a printed summary. **Check:** `DATASET_ROOT` is correct; the GPU line (without a GPU, training on ~34,000 images is slow: use `LIMIT_GROUPS_PER_CLASS` for a quick trial first).
**Edit only this cell** for settings. `CLASSES_CONFIRMED` stays `False` until you have read sections 01-02.

In [ ]:
import os, sys, io, re, json, math, time, random, shutil, hashlib, platform, datetime, warnings, itertools, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")

# ============================== CONFIGURATION (edit here) ==============================
EXPERIMENT_NAME = "fitzpatrick_mnv2_v1"                      # new name = new experiment folder (nothing is overwritten)

if os.name == "nt":
    DATASET_ROOT = Path(r"C:\Users\Bautista\Downloads\PrecisionSkin\Fitzpatrick17k")
else:                                                          # WSL
    DATASET_ROOT = Path("/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Fitzpatrick17k")
OUTPUT_ROOT = Path.home() / "PrecisionSkinV1" / "fitzpatrick_experiments" / EXPERIMENT_NAME

# Your folder names are the labels. The ORDER (Fitzpatrick type) must be stated explicitly, because ordinal metrics need it.
# It is checked against the folders that really exist (section 01); an unknown folder stops the notebook instead of being guessed.
FOLDER_TO_FITZPATRICK = {"Type I": 1, "Type II": 2, "TYPE_III": 3, "TYPE_IV": 4, "TYPE_V": 5, "TYPE_VI": 6}
EXCLUDE_FOLDERS = []             # e.g. ["TYPE_VI"] - only AFTER you saw its count in section 02. [] = use every folder found.
CLASSES_CONFIRMED = False        # set True after you read sections 01-02 and agree with the class list and counts

SEED = 42
BACKBONE = "MobileNetV2"
PRETRAINED_WEIGHTS = "imagenet"  # needs internet the first time (downloads the weights)
IMG_SIZE = 224
RESIZE_MODE = "pad"              # "pad" (keep whole image, pad), "crop" (centre square crop) or "stretch"
TRAIN_BATCH_SIZE = 32          # (section 04 has its own BATCH_SIZE for the image browser)
SPLIT = (0.70, 0.15, 0.15)       # train / validation / test, by duplicate GROUP (so photos of one group never cross splits)

# ---- duplicates / leakage ----
NEAR_DUP_MIN_CORR = 0.97         # 32x32 grey-thumbnail correlation (lower = catches more copies). A missed copy risks leakage; a wrong merge only keeps two photos together.
NEAR_DUP_VARIANTS = "dihedral8"  # compare also rotated/flipped copies ("dihedral8"), or only flips ("flip2"), or none ("none")
MAX_GROUP_SIZE = 200             # a group larger than this is flagged for review (it is still kept together in one split)
EXCLUDE_CONFLICTING_GROUPS = True   # same photo found under two different Fitzpatrick folders -> kept out of training AND evaluation
MAX_TRAIN_COPIES_PER_GROUP = 3   # near-identical copies in training are capped; validation/test always get ONE image per group
LIMIT_GROUPS_PER_CLASS = None    # None = everything. e.g. 200 for a quick trial run (the notebook then says TRIAL RUN)

# ---- augmentation (training split only). Geometric only by default; colour is NOT altered. ----
AUG = dict(hflip=True, rotation_deg=8, translate=0.05, zoom=0.10,
           photometric=False, brightness_delta=0.0)     # photometric stays off; if you ever enable it keep brightness_delta <= 0.05

# ---- training ----
EPOCHS_A, LR_A = 12, 1e-3                    # Stage A: frozen backbone
EPOCHS_B, LR_B = 25, 1e-5                    # Stage B: fine-tuning (much lower learning rate)
UNFREEZE_LAST_LAYERS = 40                    # last N backbone layers are trained in Stage B (BatchNorm layers stay frozen)
DROPOUT = 0.30
EARLY_STOP_PATIENCE = 5
REDUCE_LR_PATIENCE, REDUCE_LR_FACTOR, MIN_LR = 2, 0.3, 1e-7
MONITOR = "val_loss"                         # checkpoints / early stopping use VALIDATION only
CLASS_WEIGHT_MODE = "balanced"               # "balanced", "sqrt_balanced" or "none"  (no blind duplication of small classes)

# ---- evaluation / export ----
RUN_TEST_EVALUATION = False      # keep False until the model is final; then set True ONCE (section 16)
ANALYSIS_SPLIT = "val"           # which split the error-analysis pictures use: "val" (default) or "test" (after section 16)
RUN_GRADCAM_AUDIT = True
ONNX_OPSET = 13
VERIFY_SAMPLES_PER_CLASS = 40

LIMITATION_TEXT = ("Fitzpatrick Skin Type is not simply a visual skin-colour category. This model's output is an AI-predicted Fitzpatrick "
                   "category based on the dataset labels, not a clinical determination of a person's actual phototype. Grad-CAM++ is an "
                   "attribution method, not skin segmentation and not proof of the true Fitzpatrick type.")
# =======================================================================================

SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
ROMAN = {1: "I", 2: "II", 3: "III", 4: "IV", 5: "V", 6: "VI"}

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
random.seed(SEED); np.random.seed(SEED)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR"); tf.autograph.set_verbosity(0)

D = {k: OUTPUT_ROOT / v for k, v in dict(audit="01_audit", manifest="02_manifest", splits="03_splits", models="04_models",
                                         figures="05_figures", reports="06_reports", export="07_export").items()}
assert not str(OUTPUT_ROOT.resolve()).startswith(str(DATASET_ROOT.resolve())), "OUTPUT_ROOT must not be inside the dataset folder"
for p in D.values():
    p.mkdir(parents=True, exist_ok=True)

print("experiment      :", EXPERIMENT_NAME, "| outputs ->", OUTPUT_ROOT)
print("dataset (R/O)   :", DATASET_ROOT, "| exists:", DATASET_ROOT.exists())
print("python %s | TF %s | Keras %s | numpy %s | pandas %s" % (platform.python_version(), tf.__version__, keras.__version__, np.__version__, pd.__version__))
gpus = tf.config.list_physical_devices("GPU")
print("GPU visible     :", [g.name for g in gpus] if gpus else "NONE -> training runs on the CPU (slow; try LIMIT_GROUPS_PER_CLASS first)")
if LIMIT_GROUPS_PER_CLASS:
    print("*** TRIAL RUN: only %d duplicate-groups per class will be used ***" % LIMIT_GROUPS_PER_CLASS)

## 01 - Locate Existing Fitzpatrick17k Folders
**What it does:** looks at your dataset folder (read-only), prints the folder tree, and compares the folders it finds with `FOLDER_TO_FITZPATRICK`.
**Expect:** your six folders, with image counts and a few example file names. **Check:** every discovered folder is one you intended; nothing unexpected sits next to them. If a folder is unknown the cell stops and tells you - it never guesses a label.

In [ ]:
assert DATASET_ROOT.exists(), "Dataset folder not found: %s" % DATASET_ROOT
entries = sorted(os.scandir(DATASET_ROOT), key=lambda e: e.name.lower())
top_dirs = [e.name for e in entries if e.is_dir()]
top_files = [e.name for e in entries if e.is_file()]

print(DATASET_ROOT.name + "/")
for name in top_dirs:
    n_files, n_sub, examples = 0, 0, []
    for root, dirs, files in os.walk(DATASET_ROOT / name):
        n_files += len(files); n_sub += len(dirs)
        if len(examples) < 3:
            examples += files[: 3 - len(examples)]
    print("  |-- %-10s %7d files | %d sub-folders | e.g. %s" % (name, n_files, n_sub, examples))
if top_files:
    print("  files directly in the root (not used):", top_files[:10])

unmapped = [d for d in top_dirs if d not in FOLDER_TO_FITZPATRICK]
missing = [d for d in FOLDER_TO_FITZPATRICK if d not in top_dirs]
print("\nfolders found that are NOT in FOLDER_TO_FITZPATRICK:", unmapped or "none")
print("mapped folders that are NOT on disk                :", missing or "none")
if unmapped:
    raise RuntimeError("Unknown folder(s) %s. Add them to FOLDER_TO_FITZPATRICK with their Fitzpatrick number, or move them out. Nothing was guessed." % unmapped)
if missing:
    print("NOTE: these mapped folders do not exist and will simply be absent:", missing)

USED_FOLDERS = sorted([d for d in top_dirs if d in FOLDER_TO_FITZPATRICK and d not in EXCLUDE_FOLDERS], key=lambda d: FOLDER_TO_FITZPATRICK[d])
CLASS_NAMES = USED_FOLDERS                                   # class_index 0..K-1 in Fitzpatrick order = your folder names
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_TYPE = {i: FOLDER_TO_FITZPATRICK[c] for i, c in enumerate(CLASS_NAMES)}
NC = len(CLASS_NAMES)
print("\nCLASSES USED (class_index -> folder -> Fitzpatrick type):")
for c in CLASS_NAMES:
    print("  %d -> %-9s -> type %s" % (CLASS_TO_IDX[c], c, ROMAN[FOLDER_TO_FITZPATRICK[c]]))
if EXCLUDE_FOLDERS:
    print("EXCLUDED by your setting:", EXCLUDE_FOLDERS)

## 02 - Dataset Structure & Class Counts
**What it does:** indexes every file in the class folders (no image is opened yet): counts per class, percentages, file extensions, unsupported files, empty folders, very small files.
**Expect:** a table of counts (Type I/II are smaller than III-VI) and a bar chart. **Check before continuing:** the counts are what you expect, **Type VI's count is shown on its own line** (decide whether to keep it; to drop it set `EXCLUDE_FOLDERS = ["TYPE_VI"]` in section 00 and re-run from 00), then set `CLASSES_CONFIRMED = True`.

In [ ]:
rows = []
for folder in CLASS_NAMES:
    for root, dirs, files in os.walk(DATASET_ROOT / folder):
        for fn in files:
            p = os.path.join(root, fn)
            rows.append(dict(folder=folder, path=p, filename=fn, ext=os.path.splitext(fn)[1].lower(), size=os.path.getsize(p),
                             rel=os.path.relpath(p, DATASET_ROOT)))
idx = pd.DataFrame(rows).sort_values("path").reset_index(drop=True)
idx["supported"] = idx.ext.isin(SUPPORTED_EXT)

cnt = idx.groupby("folder").size().reindex(CLASS_NAMES)
tab = pd.DataFrame({"images": cnt, "percent": (100 * cnt / cnt.sum()).round(2)})
tab["fitzpatrick"] = [ROMAN[FOLDER_TO_FITZPATRICK[c]] for c in tab.index]
print(tab.to_string()); print("\nTOTAL images:", len(idx))
if "TYPE_VI" in CLASS_NAMES:
    print("\n>>> TYPE_VI (reported separately): %d images (%.1f %%). Keep it, or set EXCLUDE_FOLDERS=['TYPE_VI'] in section 00." % (cnt["TYPE_VI"], 100 * cnt["TYPE_VI"] / cnt.sum()))

print("\nfile extensions per class:"); print(pd.crosstab(idx.folder, idx.ext).reindex(CLASS_NAMES).to_string())
print("\nunsupported files:", int((~idx.supported).sum()), idx[~idx.supported].rel.head(10).tolist())
empty = [c for c in CLASS_NAMES if cnt.get(c, 0) == 0]
print("empty class folders:", empty or "none")
print("files < 1 KB (suspicious):", int((idx["size"] < 1024).sum()), "| zero-byte:", int((idx["size"] == 0).sum()))
print("files in sub-folders of a class folder:", int((idx.rel.str.count(r"[\\/]") > 1).sum()))
print("ratio largest/smallest class: %.2f" % (cnt.max() / cnt.min()))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(CLASS_NAMES, cnt.values, color="#4c78a8")
for i, v in enumerate(cnt.values):
    ax.text(i, v, str(v), ha="center", va="bottom", fontsize=8)
ax.set_title("Images per Fitzpatrick folder (before any audit)"); ax.set_ylabel("images"); plt.tight_layout()
fig.savefig(D["figures"] / "class_counts_raw.png", dpi=200); plt.show()
print("\nCLASSES_CONFIRMED is currently:", CLASSES_CONFIRMED)

## 03 - Dataset Integrity Audit
**What it does:** opens **every** image once (read-only) and records: readable or not, width/height, mode, MD5, a 32x32 colour thumbnail (used for duplicates) and the average colour of the central area.
It then flags suspicious files. Results are cached in `01_audit/features_cache.pkl`, so a re-run is fast.
**Expect:** a progress bar, then per-class size statistics and counts of flagged files. **Check:** unreadable files (should be 0), tiny images, odd aspect ratios.
Flagged files go to `01_audit/audit_flagged.csv` - **nothing is deleted or moved**.
The "ITA outlier" hint compares each image's central skin colour (Individual Typology Angle) with its own folder: it only suggests images worth a manual look; ITA is *not* the Fitzpatrick type and the hint can be wrong (lighting, lesion colour).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import cv2

CACHE = D["audit"] / "features_cache.pkl"
cache = pickle.load(open(CACHE, "rb")) if CACHE.exists() else {}


def extract(path):
    h = None
    try:
        with open(path, "rb") as f:
            b = f.read()
        h = hashlib.md5(b).hexdigest()
        im = Image.open(io.BytesIO(b)); w, hh = im.size; fmt, mode = im.format, im.mode
        im.load()
        rgb = im.convert("RGB")
        th = np.asarray(rgb.resize((32, 32), Image.BOX), dtype=np.uint8)
        cx, cy = w // 4, hh // 4
        cen = np.asarray(rgb.crop((cx, cy, max(cx + 1, w - cx), max(cy + 1, hh - cy))).resize((16, 16), Image.BOX), dtype=np.float32).mean((0, 1))
        return dict(ok=True, error="", md5=h, width=w, height=hh, mode=mode, fmt=fmt, thumb=th, cr=float(cen[0]), cg=float(cen[1]), cb=float(cen[2]))
    except Exception as e:
        return dict(ok=False, error=repr(e)[:160], md5=h, width=0, height=0, mode="", fmt="", thumb=np.zeros((32, 32, 3), np.uint8), cr=np.nan, cg=np.nan, cb=np.nan)


todo = []
for r in idx.itertuples():
    if not r.supported:
        continue
    key = r.path
    if key in cache and cache[key][0] == r.size and cache[key][1] == int(os.path.getmtime(r.path) * 1e9):
        continue
    todo.append(r)
print("images to read: %d (cached: %d)" % (len(todo), len(idx) - len(todo)))
if todo:
    with ThreadPoolExecutor(8) as ex:
        for r, res in zip(todo, tqdm(ex.map(lambda r: extract(r.path), todo), total=len(todo), desc="reading images")):
            cache[r.path] = (r.size, int(os.path.getmtime(r.path) * 1e9), res)
    pickle.dump(cache, open(CACHE, "wb"))

feat_rows, thumbs = [], np.zeros((len(idx), 32, 32, 3), np.uint8)
for i, r in enumerate(idx.itertuples()):
    if r.supported and r.path in cache:
        res = dict(cache[r.path][2]); thumbs[i] = res.pop("thumb"); feat_rows.append(res)
    else:
        feat_rows.append(dict(ok=False, error="unsupported file type" if not r.supported else "not read", md5=None, width=0, height=0, mode="", fmt="", cr=np.nan, cg=np.nan, cb=np.nan))
feat = pd.DataFrame(feat_rows)
THUMBS = thumbs
aud = pd.concat([idx.reset_index(drop=True), feat], axis=1)

# ITA from the central colour (hint only)
def ita_of(r, g, b):
    if not np.isfinite(r):
        return np.nan
    lab = cv2.cvtColor(np.uint8([[[round(r), round(g), round(b)]]]), cv2.COLOR_RGB2LAB)[0, 0]
    L, bb = lab[0] * 100.0 / 255.0, float(lab[2]) - 128.0
    return float(np.degrees(np.arctan2(L - 50.0, bb if abs(bb) > 1e-6 else 1e-6)))
aud["ita_deg"] = [ita_of(r.cr, r.cg, r.cb) for r in aud.itertuples()]
med = aud[aud.ok].groupby("folder").ita_deg.median(); mad = aud[aud.ok].groupby("folder").ita_deg.apply(lambda s: np.median(np.abs(s - s.median())) + 1e-6)
aud["ita_z"] = [0.6745 * (r.ita_deg - med[r.folder]) / mad[r.folder] if r.ok else np.nan for r in aud.itertuples()]

MIN_SIDE = 64
flags = []
for r in aud.itertuples():
    f = []
    if not r.supported: f.append("unsupported_file_type")
    elif not r.ok: f.append("unreadable_or_corrupted")
    else:
        if min(r.width, r.height) < MIN_SIDE: f.append("small_image(<%dpx)" % MIN_SIDE)
        if max(r.width, r.height) / max(1, min(r.width, r.height)) > 3: f.append("extreme_aspect_ratio")
        if abs(r.ita_z) > 4: f.append("ita_outlier_vs_own_folder (review hint only)")
    if r.size < 1024: f.append("file_under_1KB")
    flags.append(";".join(f))
aud["flags"] = flags
aud[aud["flags"] != ""][["folder", "rel", "width", "height", "size", "ita_deg", "ita_z", "flags", "error"]].to_csv(D["audit"] / "audit_flagged.csv", index=False)
aud.to_csv(D["audit"] / "audit_all_files.csv", index=False)

print("readable images:", int(aud.ok.sum()), "of", len(aud), "| unreadable/corrupted:", int((aud.supported & ~aud.ok).sum()), "| unsupported:", int((~aud.supported).sum()))
sz = aud[aud.ok].groupby("folder").agg(n=("ok", "size"), w_min=("width", "min"), w_med=("width", "median"), w_max=("width", "max"),
                                         h_min=("height", "min"), h_med=("height", "median"), h_max=("height", "max")).reindex(CLASS_NAMES)
sz["n_distinct_sizes"] = [aud[(aud.folder == c) & aud.ok].apply(lambda r: (r.width, r.height), axis=1).nunique() for c in CLASS_NAMES]
print("\nimage dimensions per class:"); print(sz.to_string())
print("\ncolour mode:", aud[aud.ok]["mode"].value_counts().to_dict(), "| format:", aud[aud.ok]["fmt"].value_counts().to_dict())
print("\nflag counts:"); print(aud["flags"].str.split(";").explode().replace("", np.nan).value_counts().to_string() or "none")
print("\nsaved: audit_flagged.csv / audit_all_files.csv in", D["audit"])

## 04 - Visual Inspection
**What it does:** shows images **with the folder label they sit in**. First a few random images per class, then a browser you control: change `CLASS_TO_VIEW`, `START_INDEX`, `BATCH_SIZE`, re-run the second cell and step through the whole class (for example `START_INDEX = 0, 25, 50, ...`).
**Expect:** image grids with titles like `TYPE_III | #125`. **Check:** do the images look like they belong to their folder? Note file names that look wrong (the CSVs are not changed by this cell; keep your own list and tell me which to flag).
`CLASS_TO_VIEW` accepts your folder name in any style (`Type I`, `Type_I`, `type i`).

In [ ]:
def norm_name(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def per_class_sample(df, col, n, seed=SEED, head=False):
    parts = []
    for c, gg in df.groupby(col, sort=True):
        parts.append(gg.head(n) if head else gg.sample(min(n, len(gg)), random_state=seed))
    return pd.concat(parts) if parts else df.iloc[:0]

def show_images(df, ncols=5, title=None, size=2.4):
    n = len(df); nrows = max(1, math.ceil(n / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * (size + 0.35)))
    for ax in np.atleast_1d(axs).ravel():
        ax.axis("off")
    for ax, (_, r) in zip(np.atleast_1d(axs).ravel(), df.iterrows()):
        try:
            ax.imshow(Image.open(r.path).convert("RGB"))
        except Exception as e:
            ax.text(0.5, 0.5, "unreadable", ha="center")
        ax.set_title("%s | #%s\n%s" % (r.folder, r.get("pos", ""), r.filename[:22]), fontsize=7)
    if title: fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

# (a) a few random images from EVERY class
rs = per_class_sample(aud[aud.ok], "folder", 5)
for c in CLASS_NAMES:
    sub = rs[rs.folder == c].copy(); sub["pos"] = ""
    show_images(sub, ncols=5, title="Random sample - folder: %s" % c)

In [ ]:
# (b) browse a whole class in batches - change these three values and re-run THIS cell
CLASS_TO_VIEW = "Type I"
START_INDEX = 0
BATCH_SIZE = 25

match = [c for c in CLASS_NAMES if norm_name(c) == norm_name(CLASS_TO_VIEW)]
assert match, "No class called %r. Available: %s" % (CLASS_TO_VIEW, CLASS_NAMES)
cls_df = aud[aud.folder == match[0]].reset_index(drop=True)
batch = cls_df.iloc[START_INDEX: START_INDEX + BATCH_SIZE].copy()
batch["pos"] = batch.index
print("class %s: %d images | showing %d..%d | next START_INDEX = %d" % (match[0], len(cls_df), START_INDEX, START_INDEX + len(batch) - 1, START_INDEX + BATCH_SIZE))
show_images(batch, ncols=5, title="%s  (items %d-%d of %d)" % (match[0], START_INDEX, START_INDEX + len(batch) - 1, len(cls_df)))

## 05 - Duplicate / Near-Duplicate Analysis
**What it does:** groups images that are the **same photo**: exact copies (identical MD5) and near-duplicates (resized/re-compressed/flipped/rotated copies; 32x32 thumbnail correlation >= `NEAR_DUP_MIN_CORR`, also against rotated and flipped versions). Every image gets a `duplicate_group`.
It also lists groups whose images sit in **different Fitzpatrick folders** (a contradiction in the labels) and unusually large groups.
**Expect:** counts, example pictures, two CSVs. This can take several minutes on ~34,000 images (the progress bar shows it).
**Check:** (1) look at the near-duplicate picture rows: are they really the same photo? If distinct photos are merged raise `NEAR_DUP_MIN_CORR` (e.g. 0.99); if obvious copies are missed lower it. (2) the cross-class groups: nothing is relabeled; those images are excluded from training/evaluation by default (`EXCLUDE_CONFLICTING_GROUPS`) so you can review them. (3) the group-size summary: many large groups = lots of augmented copies in the source.

In [ ]:
ok_idx = np.where(aud.ok.values)[0]
# 1) exact duplicates: same MD5
md5_first = {}
rep_of = np.full(len(aud), -1, dtype=np.int64)          # index of the first image with the same MD5
for i in ok_idx:
    rep_of[i] = md5_first.setdefault(aud.md5.iat[i], i)
reps = np.array(sorted(set(rep_of[ok_idx])))
print("images: %d | exact-unique (by MD5): %d | exact duplicates: %d" % (len(ok_idx), len(reps), len(ok_idx) - len(reps)))

# 2) near duplicates among the unique images (blocked correlation, with rotated/flipped versions)
def zn(x):
    x = x.reshape(len(x), -1).astype(np.float32); x = x - x.mean(1, keepdims=True)
    return (x / (x.std(1, keepdims=True) + 1e-6) / np.sqrt(x.shape[1])).astype(np.float32)

G = THUMBS[reps].astype(np.float32).mean(-1)               # grey 32x32
if NEAR_DUP_VARIANTS == "dihedral8":
    variants = []
    for k in range(4):
        rr = np.rot90(G, k, axes=(1, 2)); variants += [rr, rr[:, :, ::-1]]
elif NEAR_DUP_VARIANTS == "flip2":
    variants = [G, G[:, :, ::-1]]
else:
    variants = [G]
base = zn(G)
pairs = set(); B = 1024
t0 = time.time()
for vi, v in enumerate(variants):
    V = zn(np.ascontiguousarray(v))
    for i0 in tqdm(range(0, len(base), B), desc="near-duplicate search, variant %d/%d" % (vi + 1, len(variants)), leave=False):
        S = base[i0:i0 + B] @ V.T
        ii, jj = np.nonzero(S >= NEAR_DUP_MIN_CORR)
        for a, b in zip(ii + i0, jj):
            if b > a:
                pairs.add((int(a), int(b)))
print("near-duplicate pairs found: %d (%.0fs)" % (len(pairs), time.time() - t0))

parent = list(range(len(reps)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, b in pairs:
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)
root_of_rep = {int(reps[k]): find(k) for k in range(len(reps))}
grp_root = np.full(len(aud), -1, dtype=np.int64)
for i in ok_idx:
    grp_root[i] = root_of_rep[int(rep_of[i])]
uniq = {r: n for n, r in enumerate(sorted(set(grp_root[ok_idx])))}
aud["duplicate_group"] = ["G%06d" % uniq[r] if r >= 0 else "" for r in grp_root]
aud["exact_dup_of"] = [int(rep_of[i]) if (rep_of[i] >= 0 and rep_of[i] != i) else -1 for i in range(len(aud))]

dgrp = aud[aud.ok].groupby("duplicate_group").agg(n=("path", "size"), n_classes=("folder", "nunique"), classes=("folder", lambda s: ",".join(sorted(set(s)))))
print("\nduplicate groups: %d | groups with >1 image: %d | images inside such groups: %d" % (len(dgrp), int((dgrp.n > 1).sum()), int(dgrp[dgrp.n > 1].n.sum())))
print("group size distribution:"); print(dgrp.n.value_counts().sort_index().head(15).to_string())
print("largest groups:", dgrp.n.sort_values(ascending=False).head(5).tolist(), "| groups over MAX_GROUP_SIZE (%d): %d" % (MAX_GROUP_SIZE, int((dgrp.n > MAX_GROUP_SIZE).sum())))
print("\nper class: share of images that belong to a multi-image group")
inmulti = aud[aud.ok].assign(multi=lambda d: d.duplicate_group.map(dgrp.n) > 1).groupby("folder").multi.mean().reindex(CLASS_NAMES)
print((100 * inmulti).round(1).astype(str).add(" %").to_string())

cross = dgrp[dgrp.n_classes > 1].sort_values("n", ascending=False)
print("\nCROSS-CLASS groups (same photo under different Fitzpatrick folders): %d groups, %d images" % (len(cross), int(cross.n.sum())))
print(cross.head(10).to_string())
dgrp.reset_index().to_csv(D["audit"] / "duplicate_groups.csv", index=False)
aud[aud.duplicate_group.isin(cross.index)][["duplicate_group", "folder", "rel", "md5"]].sort_values(["duplicate_group", "folder"]).to_csv(D["audit"] / "cross_class_duplicates.csv", index=False)
print("\nsaved: duplicate_groups.csv, cross_class_duplicates.csv in", D["audit"])

def show_groups(gids, title, per_row=6):
    fig, axs = plt.subplots(len(gids), per_row, figsize=(per_row * 2.1, len(gids) * 2.3), squeeze=False)
    for ax in axs.ravel():
        ax.axis("off")
    for r, gid in enumerate(gids):
        sub = aud[aud.duplicate_group == gid].head(per_row)
        for c, (_, x) in enumerate(sub.iterrows()):
            axs[r, c].imshow(Image.open(x.path).convert("RGB")); axs[r, c].set_title("%s\n%s" % (x.folder, x.filename[:18]), fontsize=6)
        axs[r, 0].set_ylabel(gid)
    fig.suptitle(title, fontsize=10); plt.tight_layout(); plt.show()

near_only = dgrp[(dgrp.n > 1) & (dgrp.n_classes == 1)].index.tolist()
if near_only:
    show_groups(random.Random(SEED).sample(near_only, min(6, len(near_only))), "Examples of duplicate groups (same class) - are these really the same photo?")
if len(cross):
    show_groups(cross.index[:4].tolist(), "Examples of CROSS-CLASS groups (label conflict) - nothing is relabeled")

## 06 - Master Dataset Manifest
**What it does:** builds `master_manifest.csv`: one row per image with `image_id, filename, original_path, fitzpatrick_class, class_index, width, height, file_hash, duplicate_group, review_status` (+ extra columns). It also saves a snapshot of all settings (`experiment_config.json`) for your records.
`review_status`: `ok`, `flag_possible_wrong_folder` (ITA hint, kept), `review_large_group` (kept), `conflict_cross_class_duplicate` (excluded by default), `excluded_unreadable`, `excluded_unsupported`.
**Expect:** a status count table. **Check:** the numbers match your expectations. **This cell stops until you set `CLASSES_CONFIRMED = True`** (section 00), so Type VI and the class list are never accepted silently.

In [ ]:
if not CLASSES_CONFIRMED:
    raise RuntimeError("Set CLASSES_CONFIRMED = True in section 00 after you have read the class list and counts (sections 01-02).")

man = aud.copy()
man["image_id"] = ["FZ%06d" % i for i in range(len(man))]
man["fitzpatrick_class"] = man.folder
man["class_index"] = man.folder.map(CLASS_TO_IDX)
man["fitzpatrick_type"] = man.class_index.map(IDX_TO_TYPE)
man["original_path"] = man.path
man["file_hash"] = man.md5
g_n = man.groupby("duplicate_group").size(); g_cls = man.groupby("duplicate_group").folder.nunique()
status, reason = [], []
for r in man.itertuples():
    if not r.supported:
        status.append("excluded_unsupported"); reason.append("file type not supported")
    elif not r.ok:
        status.append("excluded_unreadable"); reason.append(r.error)
    elif g_cls.get(r.duplicate_group, 1) > 1:
        status.append("conflict_cross_class_duplicate"); reason.append("same photo found in several Fitzpatrick folders")
    elif g_n.get(r.duplicate_group, 1) > MAX_GROUP_SIZE:
        status.append("review_large_group"); reason.append("duplicate group larger than %d" % MAX_GROUP_SIZE)
    elif "ita_outlier" in r.flags:
        status.append("flag_possible_wrong_folder"); reason.append("central skin colour is an outlier for its own folder (heuristic hint; label NOT changed)")
    else:
        status.append("ok"); reason.append("")
man["review_status"] = status; man["review_reason"] = reason
man["use_in_experiment"] = man.review_status.isin(["ok", "flag_possible_wrong_folder", "review_large_group"]) | \
                           ((man.review_status == "conflict_cross_class_duplicate") & (not EXCLUDE_CONFLICTING_GROUPS))
cols = ["image_id", "filename", "original_path", "fitzpatrick_class", "class_index", "fitzpatrick_type", "width", "height", "file_hash",
        "duplicate_group", "review_status", "review_reason", "use_in_experiment", "ita_deg", "flags", "size"]
man = man[cols].rename(columns={"size": "file_size_bytes"})
man.to_csv(D["manifest"] / "master_manifest.csv", index=False)

print(pd.crosstab(man.fitzpatrick_class, man.review_status).reindex(CLASS_NAMES).to_string())
print("\nused in the experiment:", int(man.use_in_experiment.sum()), "of", len(man))
json.dump(dict(experiment=EXPERIMENT_NAME, created=datetime.datetime.now().isoformat(timespec="seconds"),
               config={k: (str(v) if isinstance(v, Path) else v) for k, v in globals().items() if k.isupper() and k not in ("LIMITATION_TEXT",)
                       and isinstance(v, (int, float, str, bool, list, tuple, dict, Path, type(None)))},
               classes=CLASS_NAMES, class_to_index=CLASS_TO_IDX, limitation=LIMITATION_TEXT),
          open(D["reports"] / "experiment_config.json", "w"), indent=2, default=str)
print("saved master_manifest.csv ->", D["manifest"], "| experiment_config.json ->", D["reports"])

## 07 - Leakage-Safe Train/Validation/Test Split
**What it does:** splits **duplicate groups** (never single images) per class with the fixed seed, so every copy of a photo lands in the same split. Validation and test get **one image per group**; training keeps up to `MAX_TRAIN_COPIES_PER_GROUP` copies. Groups with a label conflict are excluded (see 06). No patient/case ID exists in the folders, so grouping is by duplicate group only.
The **test set is frozen**: its hash is saved in `03_splits/test_frozen.json`; if a later run produces a different test set the notebook stops instead of silently changing it.
Augmentation happens **later and only on training images**.
**Expect:** a table per class and split plus "no leakage" checks. **Check:** all checks print OK; every class has validation and test images. If you change the seed or split settings, use a new `EXPERIMENT_NAME`.

In [ ]:
rng = np.random.RandomState(SEED)
use = man[man.use_in_experiment].copy()
use["pixels"] = use.width.astype(np.int64) * use.height
grp = use.groupby("duplicate_group").agg(cls=("class_index", "first"), ncls=("class_index", "nunique"), n=("image_id", "size"))
assert (grp.ncls == 1).all(), "a duplicate group has images from several classes"

group_split = {}
for c, gg in grp.groupby("cls"):
    ids = sorted(gg.index); rng.shuffle(ids)
    if LIMIT_GROUPS_PER_CLASS:
        for gid in ids[LIMIT_GROUPS_PER_CLASS:]:
            group_split[gid] = "unused_trial_limit"
        ids = ids[:LIMIT_GROUPS_PER_CLASS]
    n = len(ids); nt = max(1, int(round(n * SPLIT[2]))); nv = max(1, int(round(n * SPLIT[1])))
    assert n >= 3, "class %s has fewer than 3 duplicate groups" % CLASS_NAMES[c]
    for k, gid in enumerate(ids):
        group_split[gid] = "test" if k < nt else ("val" if k < nt + nv else "train")

use["grp_split"] = use.duplicate_group.map(group_split)
use = use.sort_values(["duplicate_group", "pixels", "image_id"], ascending=[True, False, True])
use["rank_in_group"] = use.groupby("duplicate_group").cumcount()
def final_split(r):
    if r.grp_split in ("val", "test"):
        return r.grp_split if r.rank_in_group == 0 else "unused_extra_copy"
    if r.grp_split == "train":
        return "train" if r.rank_in_group < MAX_TRAIN_COPIES_PER_GROUP else "unused_extra_copy"
    return r.grp_split
use["split"] = [final_split(r) for r in use.itertuples()]

split_df = man[["image_id", "filename", "original_path", "fitzpatrick_class", "class_index", "fitzpatrick_type", "file_hash", "duplicate_group", "review_status", "width", "height"]].merge(
    use[["image_id", "split"]], on="image_id", how="left")
split_df["split"] = split_df.split.fillna("excluded")
split_df.to_csv(D["splits"] / "split_manifest.csv", index=False)

train_df = split_df[split_df.split == "train"].reset_index(drop=True)
val_df = split_df[split_df.split == "val"].reset_index(drop=True)
test_df = split_df[split_df.split == "test"].reset_index(drop=True)

summ = pd.crosstab(split_df.fitzpatrick_class, split_df.split).reindex(CLASS_NAMES)
print(summ.to_string()); print("\nTOTAL train %d | val %d | test %d" % (len(train_df), len(val_df), len(test_df)))
# leakage checks
gs = split_df[split_df.split.isin(["train", "val", "test"])].groupby("duplicate_group").split.nunique()
hs = split_df[split_df.split.isin(["train", "val", "test"])].groupby("file_hash").split.nunique()
assert (gs == 1).all(), "LEAK: a duplicate group is in more than one split"
assert (hs == 1).all(), "LEAK: an identical file (MD5) is in more than one split"
for s, d in (("val", val_df), ("test", test_df)):
    assert d.duplicate_group.is_unique, "more than one image of a group in %s" % s
    assert set(d.class_index) == set(range(NC)), "%s is missing a class" % s
print("leakage checks: OK (no group / no identical file crosses train, val, test; one image per group in val/test)")

# frozen test set
th = hashlib.sha256("\n".join(sorted(test_df.image_id + ":" + test_df.file_hash)).encode()).hexdigest()
fz = D["splits"] / "test_frozen.json"
if fz.exists():
    old = json.load(open(fz))
    if old["sha256"] != th:
        raise RuntimeError("The TEST SET CHANGED compared with the frozen one (%s). Use a new EXPERIMENT_NAME if you really changed seed/split settings." % fz)
    print("frozen test set: unchanged since", old["created"])
else:
    json.dump(dict(sha256=th, n=len(test_df), seed=SEED, split=SPLIT, created=datetime.datetime.now().isoformat(timespec="seconds")), open(fz, "w"), indent=2)
    print("frozen test set saved:", th[:16], "...")
print("split manifest saved:", D["splits"] / "split_manifest.csv")

## 08 - Data Pipeline
**What it does:** builds `tf.data` pipelines that read the **original files** (never modifying them), decode to RGB, resize to `IMG_SIZE` (pad / crop / stretch) and keep pixel values in 0-255 (the model rescales them inside, so the exported ONNX contains the preprocessing). Only the training pipeline augments.
**Expect:** the shapes and value range of one batch. **Check:** `(32, 224, 224, 3)`, float32, values 0-255, labels are integers 0..K-1.

In [ ]:
def load_resize(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.cast(img, tf.float32)
    if RESIZE_MODE == "pad":
        img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE, method="bilinear", antialias=True)
    elif RESIZE_MODE == "crop":
        s = tf.shape(img); side = tf.minimum(s[0], s[1])
        img = tf.image.crop_to_bounding_box(img, (s[0] - side) // 2, (s[1] - side) // 2, side, side)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], antialias=True)
    else:
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], antialias=True)
    return tf.clip_by_value(img, 0.0, 255.0)

aug_layers = []
if AUG["hflip"]: aug_layers.append(layers.RandomFlip("horizontal"))
if AUG["rotation_deg"]: aug_layers.append(layers.RandomRotation(AUG["rotation_deg"] / 360.0, fill_mode="reflect"))
if AUG["translate"]: aug_layers.append(layers.RandomTranslation(AUG["translate"], AUG["translate"], fill_mode="reflect"))
if AUG["zoom"]: aug_layers.append(layers.RandomZoom(AUG["zoom"], fill_mode="reflect"))
augmenter = keras.Sequential(aug_layers, name="geometric_augmentation")

def augment_batch(x):
    x = augmenter(x, training=True)
    if AUG["photometric"] and AUG["brightness_delta"] > 0:          # conservative, off by default
        x = tf.image.random_brightness(x, AUG["brightness_delta"] * 255.0)
    return tf.clip_by_value(x, 0.0, 255.0)

def make_ds(df, training):
    ds = tf.data.Dataset.from_tensor_slices((df.original_path.values.astype(str), df.class_index.values.astype(np.int32)))
    if training:
        ds = ds.shuffle(min(len(df), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_resize(p), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(TRAIN_BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (augment_batch(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)
xb, yb = next(iter(train_ds))
print("train batch:", xb.shape, xb.dtype, "| value range %.0f - %.0f" % (float(tf.reduce_min(xb)), float(tf.reduce_max(xb))), "| labels:", yb.numpy()[:12])
print("steps per epoch:", math.ceil(len(train_df) / TRAIN_BATCH_SIZE), "| val steps:", math.ceil(len(val_df) / TRAIN_BATCH_SIZE))

## 09 - Safe Augmentation Preview
**What it does:** shows some training images next to three augmented versions, and measures how much the **average colour** changes (it should hardly change - only geometry is altered).
**Expect:** slightly rotated / shifted / zoomed / flipped images with natural colour; a small number for the colour shift. **Check:** skin colour looks the same as in the original. If anything looks unnatural, lower the values in `AUG` (section 00) and re-run from 08.

In [ ]:
sample = per_class_sample(train_df, "class_index", 1).reset_index(drop=True)
orig = tf.stack([load_resize(p) for p in sample.original_path.values])
fig, axs = plt.subplots(len(sample), 4, figsize=(8.5, 2.2 * len(sample)))
for r in range(len(sample)):
    axs[r, 0].imshow(orig[r].numpy().astype("uint8")); axs[r, 0].set_title("original | %s" % sample.fitzpatrick_class[r], fontsize=7)
    for c in range(1, 4):
        axs[r, c].imshow(augment_batch(orig[r:r + 1])[0].numpy().astype("uint8")); axs[r, c].set_title("augmented %d" % c, fontsize=7)
for ax in axs.ravel(): ax.axis("off")
plt.tight_layout(); fig.savefig(D["figures"] / "augmentation_preview.png", dpi=200); plt.show()

big = tf.stack([load_resize(p) for p in train_df.original_path.values[:64]])
aug = augment_batch(big)
shift = tf.reduce_mean(tf.abs(tf.reduce_mean(big, axis=(1, 2)) - tf.reduce_mean(aug, axis=(1, 2)))).numpy()
print("mean absolute change of the average R,G,B (0-255 scale) after augmentation: %.2f   (small = colour preserved)" % shift)
print("photometric augmentation enabled:", AUG["photometric"], "| hue/saturation/CLAHE/equalization/white-balance: not used")

## 10 - Class Imbalance Analysis
**What it does:** shows exact number and percentage of images per class in each split and computes **class weights** (a loss weighting, no blind duplication of small classes).
**Expect:** tables, a chart, and the weights. **Check:** the weights look reasonable (classes with fewer training images get larger weights). `CLASS_WEIGHT_MODE` in section 00: `balanced`, `sqrt_balanced` (gentler) or `none`.

In [ ]:
for name, d in (("TRAIN", train_df), ("VALIDATION", val_df), ("TEST", test_df)):
    c = d.groupby("fitzpatrick_class").size().reindex(CLASS_NAMES).fillna(0).astype(int)
    print(name); print(pd.DataFrame({"images": c, "percent": (100 * c / c.sum()).round(2)}).to_string()); print()

from sklearn.utils.class_weight import compute_class_weight
ytr = train_df.class_index.values
w = compute_class_weight("balanced", classes=np.arange(NC), y=ytr)
if CLASS_WEIGHT_MODE == "sqrt_balanced":
    w = np.sqrt(w)
elif CLASS_WEIGHT_MODE == "none":
    w = np.ones(NC)
CLASS_WEIGHTS = {int(i): float(x) for i, x in enumerate(w)}
print("class weights (%s):" % CLASS_WEIGHT_MODE, {CLASS_NAMES[i]: round(v, 3) for i, v in CLASS_WEIGHTS.items()})
ctr = train_df.groupby("class_index").size().reindex(range(NC))
print("imbalance ratio in training (largest / smallest): %.2f" % (ctr.max() / ctr.min()))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(CLASS_NAMES, ctr.values, color="#4c78a8"); ax[0].set_title("training images per class"); ax[0].tick_params(axis="x", rotation=30)
ax[1].bar(CLASS_NAMES, [CLASS_WEIGHTS[i] for i in range(NC)], color="#f58518"); ax[1].set_title("class weights"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); fig.savefig(D["figures"] / "class_balance.png", dpi=200); plt.show()

## 11 - MobileNetV2 Model Construction
**What it does:** builds `input (0-255 RGB) -> Rescaling to [-1,1] -> MobileNetV2 (ImageNet weights) -> GlobalAveragePooling -> Dropout -> Dense (logits) -> Softmax`. The rescaling is inside the model, so the exported ONNX accepts raw 0-255 pixels.
**Expect:** the layer list and parameter counts (about 2.3 M backbone parameters). **Check:** output shape is `(None, K)` with K = number of classes; the first run downloads the ImageNet weights (needs internet).

In [ ]:
def build_model():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="input")
    x = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="rescale_minus1_1")(inp)
    backbone = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights=PRETRAINED_WEIGHTS)
    x = backbone(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(DROPOUT, name="dropout")(x)
    x = layers.Dense(NC, name="fitz_logits")(x)
    out = layers.Softmax(name="fitz_probs")(x)
    return keras.Model(inp, out, name="fitzpatrick_mobilenetv2")

def get_backbone(m):
    return [l for l in m.layers if isinstance(l, keras.Model)][0]

model = build_model()
backbone = get_backbone(model)
print("output shape:", model.output_shape, "| classes:", CLASS_NAMES)
print("backbone layers: %d | total params %s | backbone params %s" % (len(backbone.layers), format(model.count_params(), ","), format(backbone.count_params(), ",")))
model.summary(line_length=100)

## 12 - Stage A: Frozen Training
**What it does:** freezes the ImageNet backbone and trains only the new head (learning rate `LR_A`). Callbacks: **ModelCheckpoint** (keeps the best *validation* checkpoint), **EarlyStopping**, **ReduceLROnPlateau**, **CSVLogger**. Class weights are applied.
**Expect:** epoch lines with loss/accuracy for training and validation; files `04_models/stageA_best.keras` and `stageA_log.csv`. **Check:** validation loss falls in the first epochs; large gap between training and validation = overfitting. The test set is **not** used here.

In [ ]:
class LRLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr = self.model.optimizer.learning_rate
        logs["lr"] = float(lr.numpy() if hasattr(lr, "numpy") else lr)

def callbacks_for(stage):
    return [LRLogger(),
            keras.callbacks.ModelCheckpoint(str(D["models"] / ("stage%s_best.keras" % stage)), monitor=MONITOR, save_best_only=True, verbose=1),
            keras.callbacks.EarlyStopping(monitor=MONITOR, patience=EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor=MONITOR, factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=1),
            keras.callbacks.CSVLogger(str(D["models"] / ("stage%s_log.csv" % stage)), append=False)]

backbone.trainable = False
model.compile(optimizer=keras.optimizers.Adam(LR_A), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("trainable parameters (Stage A):", format(sum(int(np.prod(v.shape)) for v in model.trainable_variables), ","))
t0 = time.time()
hist_A = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_A, class_weight=CLASS_WEIGHTS, callbacks=callbacks_for("A"), verbose=2)
print("Stage A finished in %.1f min | best %s = %.4f" % ((time.time() - t0) / 60, MONITOR, min(hist_A.history[MONITOR])))

## 13 - Stage B: Fine-Tuning
**What it does:** reloads the best Stage A checkpoint, unfreezes the **last `UNFREEZE_LAST_LAYERS` backbone layers** (BatchNorm layers stay frozen so their statistics are not disturbed) and continues with a much lower learning rate `LR_B`. Same callbacks; files `stageB_best.keras`, `stageB_log.csv`.
**Expect:** validation loss improves slowly, or Stage B stops early. **Check:** if Stage B does not beat Stage A the next section still keeps whichever is better on **validation**.

In [ ]:
model = keras.models.load_model(D["models"] / "stageA_best.keras")
backbone = get_backbone(model)
backbone.trainable = True
for l in backbone.layers[:-UNFREEZE_LAST_LAYERS]:
    l.trainable = False
for l in backbone.layers[-UNFREEZE_LAST_LAYERS:]:
    l.trainable = not isinstance(l, layers.BatchNormalization)
model.compile(optimizer=keras.optimizers.Adam(LR_B), loss="sparse_categorical_crossentropy", metrics=["accuracy"])    # must recompile after changing trainable
print("trainable parameters (Stage B):", format(sum(int(np.prod(v.shape)) for v in model.trainable_variables), ","))
t0 = time.time()
hist_B = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_B, class_weight=CLASS_WEIGHTS, callbacks=callbacks_for("B"), verbose=2)
print("Stage B finished in %.1f min | best %s = %.4f" % ((time.time() - t0) / 60, MONITOR, min(hist_B.history[MONITOR])))

## 14 - Training Curves
**What it does:** reads the two CSV logs and saves publication-quality figures: loss, accuracy, learning rate (Stage A and Stage B, with the boundary marked).
**Expect:** three panels. **Check:** validation curves flatten or rise while training keeps improving = overfitting; the learning-rate panel shows when the scheduler reduced it.

In [ ]:
la = pd.read_csv(D["models"] / "stageA_log.csv"); lb = pd.read_csv(D["models"] / "stageB_log.csv")
la["stage"] = "A"; lb["stage"] = "B"
lb["epoch"] = lb["epoch"] + len(la)
hist = pd.concat([la, lb], ignore_index=True); hist["epoch_global"] = hist["epoch"] + (1 if la["epoch"].min() == 0 else 0)
hist.to_csv(D["reports"] / "training_history_all.csv", index=False)
bound = len(la) + 0.5
fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (a, b, t) in zip(axs[:2], [("loss", "val_loss", "Loss"), ("accuracy", "val_accuracy", "Accuracy")]):
    ax.plot(hist.epoch_global, hist[a], label="train", lw=2); ax.plot(hist.epoch_global, hist[b], label="validation", lw=2)
    ax.axvline(bound, color="grey", ls="--", lw=1); ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=.3)
axs[2].semilogy(hist.epoch_global, hist.lr, lw=2, color="#54a24b"); axs[2].axvline(bound, color="grey", ls="--", lw=1)
axs[2].set_title("Learning rate"); axs[2].set_xlabel("epoch"); axs[2].grid(alpha=.3)
fig.suptitle("Training history (dashed line = start of Stage B fine-tuning)", fontsize=10); plt.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(D["figures"] / ("training_curves." + ext), dpi=220)
plt.show()
print("best validation loss: %.4f at global epoch %d" % (hist.val_loss.min(), hist.loc[hist.val_loss.idxmin(), "epoch_global"]))

## 15 - Validation Evaluation
**What it does:** picks the **final model = the better of the Stage A / Stage B checkpoints by validation loss** (validation only; the test set plays no role), then computes the full metric set on the validation split: accuracy, balanced accuracy, macro precision/recall/F1, weighted F1, Cohen's kappa, quadratic weighted kappa, per-class precision/recall/F1/support.
**Expect:** a metrics table and per-class table; `06_reports/val_metrics.json`, `val_predictions.csv` (with all class scores). **Check:** balanced accuracy and macro F1 (not just accuracy); weak classes.

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, f1_score, cohen_kappa_score,
                             confusion_matrix)

best_loss = {}
for stage in ("A", "B"):
    m = keras.models.load_model(D["models"] / ("stage%s_best.keras" % stage))
    m.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    best_loss[stage] = m.evaluate(val_ds, verbose=0)[0]
print("validation loss of the checkpoints:", {k: round(v, 4) for k, v in best_loss.items()})
BEST_STAGE = min(best_loss, key=best_loss.get)
shutil.copy2(D["models"] / ("stage%s_best.keras" % BEST_STAGE), D["models"] / "best_validation_model.keras")
model = keras.models.load_model(D["models"] / "best_validation_model.keras")
print("FINAL MODEL = Stage %s checkpoint (chosen on validation loss only)" % BEST_STAGE)

TYPES = np.array([IDX_TO_TYPE[i] for i in range(NC)])

def predict_probs(m, df):
    return m.predict(make_ds(df, False), verbose=0)

def evaluate_predictions(y_true, probs):
    y_pred = probs.argmax(1); labels = list(range(NC))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    dist = np.abs(TYPES[y_true] - TYPES[y_pred])
    res = dict(n=int(len(y_true)), accuracy=accuracy_score(y_true, y_pred), balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
               macro_precision=float(p.mean()), macro_recall=float(r.mean()), macro_f1=float(f.mean()),
               weighted_f1=float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
               cohen_kappa=float(cohen_kappa_score(y_true, y_pred, labels=labels)),
               quadratic_weighted_kappa=float(cohen_kappa_score(y_true, y_pred, labels=labels, weights="quadratic")),
               mean_abs_class_distance=float(dist.mean()), within_1_type=float((dist <= 1).mean()), exact=float((dist == 0).mean()),
               severe_error_rate_2plus_types=float((dist >= 2).mean()))
    per = pd.DataFrame(dict(precision=p, recall=r, f1=f, support=s), index=CLASS_NAMES)
    return res, per, y_pred, dist

def prediction_table(df, probs):
    t = df[["image_id", "filename", "original_path", "fitzpatrick_class", "class_index", "duplicate_group"]].copy()
    t["pred_index"] = probs.argmax(1); t["pred_class"] = [CLASS_NAMES[i] for i in t.pred_index]; t["confidence"] = probs.max(1)
    t["class_distance"] = np.abs(TYPES[t.class_index.values] - TYPES[t.pred_index.values])
    for i, c in enumerate(CLASS_NAMES):
        t["score_" + c] = probs[:, i]
    return t

probs_val = predict_probs(model, val_df)
val_res, val_per, val_pred, val_dist = evaluate_predictions(val_df.class_index.values, probs_val)
val_tab = prediction_table(val_df, probs_val); val_tab.to_csv(D["reports"] / "val_predictions.csv", index=False)
json.dump(val_res, open(D["reports"] / "val_metrics.json", "w"), indent=2); val_per.to_csv(D["reports"] / "val_per_class.csv")
print("\nVALIDATION metrics (n=%d)" % val_res["n"]); print(pd.Series(val_res).drop("n").round(4).to_string())
print("\nper-class:"); print(val_per.round(4).to_string())

## 16 - Frozen Test Evaluation
**What it does:** evaluates the **final model once** on the frozen test set after verifying the test hash. It is blocked until you set `RUN_TEST_EVALUATION = True` (section 00) - do that only when you will no longer change the model, epochs, hyperparameters, architecture, seed or thresholds.
**Expect:** the same metric set as section 15 on unseen photos; `test_metrics.json`, `test_predictions.csv`. **Check:** test numbers are usually a little below validation. **Do not tune anything based on these numbers.**

In [ ]:
TEST_RES = None
if not RUN_TEST_EVALUATION:
    print("Test evaluation is switched off (RUN_TEST_EVALUATION = False). Finish all validation work first, then set it to True in section 00 and run this cell.")
else:
    fz = json.load(open(D["splits"] / "test_frozen.json"))
    th = hashlib.sha256("\n".join(sorted(test_df.image_id + ":" + test_df.file_hash)).encode()).hexdigest()
    assert th == fz["sha256"], "test set differs from the frozen one"
    probs_test = predict_probs(model, test_df)
    TEST_RES, test_per, test_pred, test_dist = evaluate_predictions(test_df.class_index.values, probs_test)
    test_tab = prediction_table(test_df, probs_test); test_tab.to_csv(D["reports"] / "test_predictions.csv", index=False)
    json.dump(TEST_RES, open(D["reports"] / "test_metrics.json", "w"), indent=2); test_per.to_csv(D["reports"] / "test_per_class.csv")
    json.dump(dict(evaluated_at=datetime.datetime.now().isoformat(timespec="seconds"), final_model_stage=BEST_STAGE, test_sha256=th),
              open(D["reports"] / "test_evaluated.json", "w"), indent=2)
    print("TEST metrics (n=%d)" % TEST_RES["n"]); print(pd.Series(TEST_RES).drop("n").round(4).to_string())
    print("\nper-class:"); print(test_per.round(4).to_string())

## 17 - Confusion Matrix & Per-Class Metrics
**What it does:** draws the confusion matrix (counts) and the row-normalized matrix, plus a per-class precision/recall/F1 bar chart, for validation (and test if section 16 was run). Figures are saved as PNG.
**Expect:** a diagonal-heavy matrix. **Check:** where the off-diagonal mass sits: neighbouring types (II<->III) are less serious than distant ones (I<->V); the normalized matrix shows each class's recall on the diagonal.

In [ ]:
def plot_confusions(y_true, y_pred, tag):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NC))); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    for ax, M, fmt, t in ((axs[0], cm, "d", "Confusion matrix (counts)"), (axs[1], cmn, ".2f", "Normalized (rows sum to 1)")):
        im = ax.imshow(M, cmap="Blues"); ax.set_title("%s - %s" % (t, tag)); ax.set_xlabel("predicted"); ax.set_ylabel("actual")
        ax.set_xticks(range(NC)); ax.set_xticklabels(CLASS_NAMES, rotation=40, ha="right"); ax.set_yticks(range(NC)); ax.set_yticklabels(CLASS_NAMES)
        for i, j in itertools.product(range(NC), range(NC)):
            ax.text(j, i, format(M[i, j], fmt), ha="center", va="center", color="white" if M[i, j] > M.max() / 2 else "black", fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); fig.savefig(D["figures"] / ("confusion_matrix_%s.png" % tag), dpi=220); plt.show()
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(D["reports"] / ("confusion_matrix_%s.csv" % tag))

def plot_per_class(per, tag):
    fig, ax = plt.subplots(figsize=(8, 3.6)); xx = np.arange(NC); w = 0.26
    for k, (m, col) in enumerate((("precision", "#4c78a8"), ("recall", "#f58518"), ("f1", "#54a24b"))):
        ax.bar(xx + (k - 1) * w, per[m].values, w, label=m, color=col)
    ax.set_xticks(xx); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right"); ax.set_ylim(0, 1); ax.legend(); ax.grid(axis="y", alpha=.3)
    ax.set_title("Per-class performance - %s" % tag); plt.tight_layout(); fig.savefig(D["figures"] / ("per_class_%s.png" % tag), dpi=220); plt.show()

plot_confusions(val_df.class_index.values, val_pred, "validation"); plot_per_class(val_per, "validation")
if TEST_RES is not None:
    plot_confusions(test_df.class_index.values, test_pred, "test"); plot_per_class(test_per, "test")

## 18 - Ordinal Error Analysis
**What it does:** because Fitzpatrick types are ordered, it measures **how far** predictions are from the truth: mean absolute class distance, share within +-1 type, quadratic weighted kappa, and the distribution of distances (0, 1, 2, 3+ types). A `Type II -> Type III` error counts as 1, `Type I -> Type V` as 4.
**Expect:** a distance table and heat-map. **Check:** most errors should be distance 1; many errors at distance >= 2 mean the model does not order skin tones sensibly. Also look at the bias: a class whose mean prediction is shifted up or down.

In [ ]:
def ordinal_report(df, y_pred, dist, res, tag):
    print("== %s ==" % tag)
    print("mean absolute class-distance error: %.3f types | within +-1 type: %.1f %% | exact: %.1f %% | distance>=2: %.1f %% | QWK: %.3f" % (
        res["mean_abs_class_distance"], 100 * res["within_1_type"], 100 * res["exact"], 100 * res["severe_error_rate_2plus_types"], res["quadratic_weighted_kappa"]))
    dd = pd.Series(np.minimum(dist, 3)).map({0: "0 (correct)", 1: "1 type off", 2: "2 types off", 3: "3+ types off"}).value_counts().reindex(["0 (correct)", "1 type off", "2 types off", "3+ types off"]).fillna(0).astype(int)
    print(pd.DataFrame({"images": dd, "percent": (100 * dd / dd.sum()).round(2)}).to_string())
    per_cls = pd.DataFrame({"actual": CLASS_NAMES,
                            "mean_abs_distance": [dist[df.class_index.values == i].mean() if (df.class_index.values == i).any() else np.nan for i in range(NC)],
                            "mean_signed_error(pred-true types)": [(TYPES[y_pred[df.class_index.values == i]] - TYPES[i]).mean() if (df.class_index.values == i).any() else np.nan for i in range(NC)]}).set_index("actual")
    print(per_cls.round(3).to_string())
    per_cls.to_csv(D["reports"] / ("ordinal_per_class_%s.csv" % tag))
    ct = pd.crosstab(df.fitzpatrick_class.values, np.minimum(dist, 4)).reindex(CLASS_NAMES).fillna(0)
    fig, ax = plt.subplots(figsize=(6.5, 3.6)); im = ax.imshow(ct.values / np.maximum(ct.values.sum(1, keepdims=True), 1), cmap="Reds")
    ax.set_yticks(range(NC)); ax.set_yticklabels(CLASS_NAMES); ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels([str(c) if c < 4 else "4+" for c in ct.columns])
    ax.set_xlabel("class distance (Fitzpatrick types)"); ax.set_title("Share of each actual class by error distance - %s" % tag); plt.colorbar(im); plt.tight_layout()
    fig.savefig(D["figures"] / ("ordinal_distance_%s.png" % tag), dpi=220); plt.show()

ordinal_report(val_df, val_pred, val_dist, val_res, "validation")
if TEST_RES is not None:
    ordinal_report(test_df, test_pred, test_dist, TEST_RES, "test")

## 19 - Misclassification Inspection
**What it does:** shows real images (read-only) for: correct predictions, incorrect predictions, **high-confidence errors**, **low-confidence** predictions, **adjacent-class** confusions (1 type off) and **severe** distance errors (2+ types). Each picture lists the actual type, the predicted type, the model's score and all class scores.
Which split: `ANALYSIS_SPLIT` in section 00 (`"val"` default; `"test"` only after section 16).
**Expect:** galleries per category. **Check:** why does it fail - lighting, filter colour, body site, tiny images, label doubt? Write down patterns; images you think are in the wrong folder: note the file name (they will not be changed automatically).

In [ ]:
HIGH_CONF, LOW_CONF, N_SHOW = 0.80, 0.45, 10
if ANALYSIS_SPLIT == "test":
    assert TEST_RES is not None, "run section 16 first (RUN_TEST_EVALUATION = True) or use ANALYSIS_SPLIT = 'val'"
    A_tab = test_tab.copy()
else:
    A_tab = val_tab.copy()
score_cols = ["score_" + c for c in CLASS_NAMES]
print("analysis split:", ANALYSIS_SPLIT, "| images:", len(A_tab))

def gallery(mask, title, n=N_SHOW, seed=SEED, ncols=5):
    sub = A_tab[mask]
    if len(sub) == 0:
        print("-- %s: none" % title); return
    sub = sub.sample(min(n, len(sub)), random_state=seed)
    nrows = math.ceil(len(sub) / ncols); fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 2.9, nrows * 3.5))
    for ax in np.atleast_1d(axs).ravel(): ax.axis("off")
    for ax, (_, r) in zip(np.atleast_1d(axs).ravel(), sub.iterrows()):
        ax.imshow(Image.open(r.original_path).convert("RGB"))
        sc = " ".join("%s:%.2f" % (c.replace("TYPE_", "").replace("Type ", "").replace("Type_", ""), r[s]) for c, s in zip(CLASS_NAMES, score_cols))
        ax.set_title("Actual: %s\nPred: %s  (score %.2f)\n%s" % (r.fitzpatrick_class, r.pred_class, r.confidence, sc), fontsize=6.3,
                     color="green" if r.class_distance == 0 else ("darkorange" if r.class_distance == 1 else "red"))
    fig.suptitle("%s  (%d of %d shown)" % (title, len(sub), int(mask.sum())), fontsize=10); plt.tight_layout(); plt.show()

right, wrong = A_tab.class_distance == 0, A_tab.class_distance > 0
gallery(right, "Correct predictions")
gallery(wrong, "Incorrect predictions")
gallery(wrong & (A_tab.confidence >= HIGH_CONF), "High-confidence errors (score >= %.2f)" % HIGH_CONF)
gallery(A_tab.confidence <= LOW_CONF, "Low-confidence predictions (top score <= %.2f)" % LOW_CONF)
gallery(A_tab.class_distance == 1, "Adjacent-class confusions (1 type off)")
gallery(A_tab.class_distance >= 2, "Severe class-distance errors (2+ types off)")
A_tab[wrong].sort_values("confidence", ascending=False).to_csv(D["reports"] / ("errors_for_manual_review_%s.csv" % ANALYSIS_SPLIT), index=False)
print("list of all errors (most confident first) saved: errors_for_manual_review_%s.csv" % ANALYSIS_SPLIT)

## 20 - Optional Grad-CAM++ Audit
**What it does:** shows, for a few correct and incorrect images, which image regions most increased the predicted class's score (Grad-CAM++ on the last convolutional features of MobileNetV2, gradients taken on the class **logit** of the predicted class).
**Important:** Grad-CAM++ is an **attribution method**. It is **not** skin segmentation, and it is **not proof of the person's true Fitzpatrick type**. Heat on hair, background, clothing or a lesion would be a warning sign; heat on skin does not make a prediction correct.
**Expect:** original and overlay pairs. **Check:** is the model looking at skin? Is it looking at borders, backgrounds or lesion colour? Switch it off with `RUN_GRADCAM_AUDIT = False`.

In [ ]:
def gradcampp(img255, class_index=None):
    x = tf.convert_to_tensor(img255[None], tf.float32)
    bb = get_backbone(model); names = [l.name for l in model.layers]; head = model.layers[names.index(bb.name) + 1:]
    resc = [l for l in model.layers if l.name == "rescale_minus1_1"][0]
    with tf.GradientTape() as tape:
        feats = bb(resc(x), training=False); tape.watch(feats)
        y = feats
        for l in head:
            if l.name == "fitz_probs":
                logits = y
            y = l(y, training=False)
        probs = y
        c = int(tf.argmax(probs[0])) if class_index is None else class_index
        score = logits[0, c]
    gr = tape.gradient(score, feats)[0].numpy(); A = feats[0].numpy()
    g2, g3 = gr ** 2, gr ** 3
    denom = 2 * g2 + A.sum((0, 1), keepdims=True) * g3
    denom = np.where(np.abs(denom) > 1e-8 * (np.abs(g2).max() + 1e-12), denom, 1e-8 * (np.abs(g2).max() + 1e-12))
    alpha = g2 / denom
    wgt = (alpha * np.maximum(gr, 0)).sum((0, 1))
    cam = np.maximum((wgt * A).sum(-1), 0)
    cam = cam / cam.max() if cam.max() > 0 else cam
    return cam, probs.numpy()[0], c

if not RUN_GRADCAM_AUDIT:
    print("Grad-CAM++ audit switched off.")
else:
    pick = pd.concat([A_tab[A_tab.class_distance == 0].sample(min(4, int((A_tab.class_distance == 0).sum())), random_state=SEED),
                      A_tab[A_tab.class_distance > 0].sample(min(4, int((A_tab.class_distance > 0).sum())), random_state=SEED)])
    fig, axs = plt.subplots(2, len(pick), figsize=(len(pick) * 2.6, 5.6), squeeze=False)
    for k, (_, r) in enumerate(pick.iterrows()):
        im = load_resize(r.original_path).numpy()
        cam, pr, c = gradcampp(im)
        heat = cv2.applyColorMap((cv2.resize(cam, (IMG_SIZE, IMG_SIZE)) * 255).astype("uint8"), cv2.COLORMAP_JET)[:, :, ::-1]
        axs[0, k].imshow(im.astype("uint8")); axs[0, k].set_title("Actual %s\nPred %s (%.2f)" % (r.fitzpatrick_class, CLASS_NAMES[c], pr[c]), fontsize=7, color="green" if r.class_distance == 0 else "red")
        axs[1, k].imshow((0.6 * im + 0.4 * heat).astype("uint8")); axs[1, k].set_title("Grad-CAM++", fontsize=7)
    for ax in axs.ravel(): ax.axis("off")
    fig.suptitle("Grad-CAM++ = attribution only; NOT skin segmentation and NOT proof of the true Fitzpatrick type", fontsize=9, color="darkred")
    plt.tight_layout(); fig.savefig(D["figures"] / "gradcampp_audit.png", dpi=200); plt.show()

## 21 - Save Final Keras Model
**What it does:** copies the validation-best model to `07_export/fitzpatrick_mobilenetv2.keras`, re-loads it from disk and checks that the saved file gives the same predictions, and records its SHA-256 checksum.
**Expect:** "identical". **Check:** the printed stage and checksum; keep them with your records.

In [ ]:
EXPORT = D["export"]
KERAS_PATH = EXPORT / "fitzpatrick_mobilenetv2.keras"
shutil.copy2(D["models"] / "best_validation_model.keras", KERAS_PATH)
reload_m = keras.models.load_model(KERAS_PATH)
chk = val_df.head(64)
pa = model.predict(make_ds(chk, False), verbose=0); pb = reload_m.predict(make_ds(chk, False), verbose=0)
print("saved model reproduces predictions: max abs difference = %.2e" % np.abs(pa - pb).max())

def sha256_of(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()
print("final model: Stage %s checkpoint | %s | %.1f MB | SHA-256 %s" % (BEST_STAGE, KERAS_PATH.name, KERAS_PATH.stat().st_size / 1e6, sha256_of(KERAS_PATH)))

## 22 - ONNX Export
**What it does:** converts the Keras model to ONNX (opset `ONNX_OPSET`) with a fixed input `(N, 224, 224, 3)`, float32, raw 0-255 RGB (the rescaling is inside the model), and loads it with ONNX Runtime to show its input/output names.
**Expect:** `fitzpatrick_mobilenetv2.onnx` and its input/output description. **Check:** do not trust the file yet - section 23 verifies the numbers. If the first converter fails the cell tries a second method and tells you which one worked.

In [ ]:
import onnx, onnxruntime as ort
ONNX_PATH = EXPORT / "fitzpatrick_mobilenetv2.onnx"
export_model = keras.models.load_model(KERAS_PATH)
spec = [tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32, name="input")]
method = None
try:
    import tf2onnx
    @tf.function(input_signature=spec)
    def serve(x):
        return export_model(x, training=False)
    tf2onnx.convert.from_function(serve, input_signature=spec, opset=ONNX_OPSET, output_path=str(ONNX_PATH))
    method = "tf2onnx.convert.from_function"
except Exception as e:
    print("method 1 failed:", repr(e)[:300])
    try:
        sm = EXPORT / "_savedmodel_tmp"
        if sm.exists(): shutil.rmtree(sm)
        export_model.export(str(sm))
        import subprocess
        subprocess.run([sys.executable, "-m", "tf2onnx.convert", "--saved-model", str(sm), "--opset", str(ONNX_OPSET), "--output", str(ONNX_PATH)], check=True)
        shutil.rmtree(sm, ignore_errors=True); method = "SavedModel + tf2onnx CLI"
    except Exception as e2:
        raise RuntimeError("ONNX export failed with both methods: %r" % e2)
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
ONNX_IN, ONNX_OUT = sess.get_inputs()[0], sess.get_outputs()[0]
print("exported with:", method)
print("ONNX input :", ONNX_IN.name, ONNX_IN.shape, ONNX_IN.type)
print("ONNX output:", ONNX_OUT.name, ONNX_OUT.shape, ONNX_OUT.type)
print("size: %.1f MB | SHA-256 %s" % (ONNX_PATH.stat().st_size / 1e6, sha256_of(ONNX_PATH)))

## 23 - Keras vs ONNX Verification
**What it does:** runs the **same images** through the Keras model and ONNX Runtime and compares the output vectors: prediction agreement, maximum and mean absolute difference. It uses up to `VERIFY_SAMPLES_PER_CLASS` validation images per class (and test images too if section 16 was run).
**Expect:** agreement 100 % (or very close) and differences around 1e-6 - 1e-4. **Check:** if agreement is below 100 % or the maximum difference is above ~1e-3, **do not deploy**; tell me the numbers.

In [ ]:
vs = per_class_sample(val_df, "class_index", VERIFY_SAMPLES_PER_CLASS, head=True).reset_index(drop=True)
if TEST_RES is not None:
    vs = pd.concat([vs, per_class_sample(test_df, "class_index", VERIFY_SAMPLES_PER_CLASS, head=True)]).reset_index(drop=True)
X = np.stack([load_resize(p).numpy() for p in vs.original_path.values]).astype(np.float32)
pk = export_model.predict(X, batch_size=TRAIN_BATCH_SIZE, verbose=0)
po = np.concatenate([sess.run([ONNX_OUT.name], {ONNX_IN.name: X[i:i + TRAIN_BATCH_SIZE]})[0] for i in range(0, len(X), TRAIN_BATCH_SIZE)])
agree = float((pk.argmax(1) == po.argmax(1)).mean())
ver = dict(n_images=int(len(X)), prediction_agreement=agree, max_abs_diff=float(np.abs(pk - po).max()), mean_abs_diff=float(np.abs(pk - po).mean()),
           onnx_probs_sum_to_one=bool(np.allclose(po.sum(1), 1, atol=1e-4)))
json.dump(ver, open(D["reports"] / "onnx_verification.json", "w"), indent=2)
print("images compared             :", ver["n_images"])
print("prediction agreement        : %.2f %%" % (100 * agree))
print("max  absolute output diff   : %.3e" % ver["max_abs_diff"])
print("mean absolute output diff   : %.3e" % ver["mean_abs_diff"])
print("ONNX outputs are probabilities (sum to 1):", ver["onnx_probs_sum_to_one"])
print("VERDICT:", "ONNX matches Keras" if agree >= 0.999 and ver["max_abs_diff"] < 1e-3 else "DO NOT DEPLOY - ONNX differs from Keras")

## 24 - Generate Deployment Metadata
**What it does:** writes `labels.json` (exactly the class-index mapping used in training, and a check that re-reads it), `model_metadata.json`, `preprocessing.json`, `deployment_summary.json`, copies the evaluation results, split manifest, master manifest and dataset audit into `07_export/records/`, and records checksums.
**Expect:** a list of files. **Check:** the labels check says identical; the limitation statement is present in the metadata.
**Integration reminder:** this model is separate from the four-class disease classifier; its output is only combined with the disease result in the WPF UI / database / report. The existing 4-class ONNX is not touched.

In [ ]:
labels = {"class_index_to_label": {str(i): CLASS_NAMES[i] for i in range(NC)},
          "class_index_to_fitzpatrick_type": {str(i): int(IDX_TO_TYPE[i]) for i in range(NC)},
          "class_index_to_roman": {str(i): ROMAN[IDX_TO_TYPE[i]] for i in range(NC)},
          "order": "ascending Fitzpatrick type", "num_classes": NC}
json.dump(labels, open(EXPORT / "labels.json", "w"), indent=2)
back = json.load(open(EXPORT / "labels.json"))
assert [back["class_index_to_label"][str(i)] for i in range(NC)] == CLASS_NAMES == [c for c, _ in sorted(CLASS_TO_IDX.items(), key=lambda kv: kv[1])]
print("labels.json is identical to the class-index mapping used in training:", CLASS_NAMES)

preproc = dict(color_order="RGB", layout="NHWC", input_shape=[None, IMG_SIZE, IMG_SIZE, 3], dtype="float32", input_value_range="0-255 (raw pixels)",
               scaling_inside_model="Rescaling(1/127.5, offset=-1) -> [-1, 1] (MobileNetV2 convention)", resize_mode=RESIZE_MODE, resize_method="bilinear, antialias",
               pad_value_if_pad=0, augmentation_at_inference=False, color_alteration="none (no CLAHE / histogram equalization / white balance / colour jitter)",
               onnx_input_name=ONNX_IN.name, onnx_output_name=ONNX_OUT.name, output="softmax probabilities, order = labels.json")
json.dump(preproc, open(EXPORT / "preprocessing.json", "w"), indent=2)

counts = {s: split_df[split_df.split == s].groupby("fitzpatrick_class").size().reindex(CLASS_NAMES).fillna(0).astype(int).to_dict() for s in ("train", "val", "test")}
meta = dict(model_name="PrecisionSkin Fitzpatrick skin-type model", experiment=EXPERIMENT_NAME, created=datetime.datetime.now().isoformat(timespec="seconds"),
            backbone=BACKBONE, pretrained_weights=PRETRAINED_WEIGHTS, img_size=IMG_SIZE, classes=CLASS_NAMES, num_classes=NC,
            final_checkpoint_stage=BEST_STAGE, selection_rule="lowest validation loss (test set not used for any choice)",
            seed=SEED, split=SPLIT, split_unit="duplicate group", images_per_split_and_class=counts,
            training=dict(stage_a=dict(epochs_max=EPOCHS_A, lr=LR_A), stage_b=dict(epochs_max=EPOCHS_B, lr=LR_B, unfrozen_last_backbone_layers=UNFREEZE_LAST_LAYERS),
                          class_weight_mode=CLASS_WEIGHT_MODE, class_weights={CLASS_NAMES[i]: v for i, v in CLASS_WEIGHTS.items()},
                          early_stopping_patience=EARLY_STOP_PATIENCE, batch_size=TRAIN_BATCH_SIZE, augmentation=AUG, trial_run=bool(LIMIT_GROUPS_PER_CLASS)),
            validation_metrics=val_res, test_metrics=TEST_RES, test_evaluated=TEST_RES is not None, onnx_verification=ver,
            dataset=dict(source_folder_name=DATASET_ROOT.name, total_images_indexed=int(len(man)), used_in_experiment=int(man.use_in_experiment.sum()),
                         conflicting_duplicate_groups_excluded=bool(EXCLUDE_CONFLICTING_GROUPS), near_duplicate_threshold=NEAR_DUP_MIN_CORR),
            versions=dict(python=platform.python_version(), tensorflow=tf.__version__, keras=keras.__version__, onnx=onnx.__version__, onnxruntime=ort.__version__),
            limitation=LIMITATION_TEXT,
            domain_note="Training images are skin photographs from a public dermatology-style dataset (varied body sites); webcam face captures are a different domain.",
            integration=dict(separate_from="PrecisionSkin four-class disease classifier (Acne / Hyperpigmentation / Eczema / Normal) - not modified, no Fitzpatrick output added to it",
                             combine_at="WPF UI, database, patient record and research report only"))
json.dump(meta, open(EXPORT / "model_metadata.json", "w"), indent=2, default=str)

REC = EXPORT / "records"; REC.mkdir(exist_ok=True)
for src, dst in ((D["manifest"] / "master_manifest.csv", "master_manifest.csv"), (D["splits"] / "split_manifest.csv", "split_manifest.csv"),
                 (D["splits"] / "test_frozen.json", "test_frozen.json"), (D["audit"] / "audit_flagged.csv", "dataset_audit_flagged.csv"),
                 (D["audit"] / "duplicate_groups.csv", "duplicate_groups.csv"), (D["audit"] / "cross_class_duplicates.csv", "cross_class_duplicates.csv"),
                 (D["reports"] / "experiment_config.json", "experiment_config.json"), (D["reports"] / "val_metrics.json", "val_metrics.json"),
                 (D["reports"] / "val_per_class.csv", "val_per_class.csv"), (D["reports"] / "training_history_all.csv", "training_history_all.csv"),
                 (D["reports"] / "onnx_verification.json", "onnx_verification.json"), (D["reports"] / "test_metrics.json", "test_metrics.json"),
                 (D["reports"] / "test_per_class.csv", "test_per_class.csv")):
    if Path(src).exists():
        shutil.copy2(src, REC / dst)

dep = dict(files={p.name: dict(sha256=sha256_of(p), bytes=p.stat().st_size) for p in (KERAS_PATH, ONNX_PATH, EXPORT / "labels.json", EXPORT / "preprocessing.json", EXPORT / "model_metadata.json")},
           how_to_use=dict(input="float32 NHWC (N,%d,%d,3), RGB, raw pixel values 0-255, resized with %s" % (IMG_SIZE, IMG_SIZE, RESIZE_MODE),
                           output="softmax over the classes of labels.json", display_wording="AI-predicted Fitzpatrick category (based on dataset labels)"),
           onnx_agreement_with_keras=ver["prediction_agreement"], ready_to_deploy=bool(ver["prediction_agreement"] >= 0.999 and ver["max_abs_diff"] < 1e-3),
           test_evaluated=TEST_RES is not None, trial_run=bool(LIMIT_GROUPS_PER_CLASS), limitation=LIMITATION_TEXT)
json.dump(dep, open(EXPORT / "deployment_summary.json", "w"), indent=2)
print("export folder:", EXPORT)
for p in sorted(EXPORT.iterdir()):
    print("  %-34s %s" % (p.name, ("%.1f MB" % (p.stat().st_size / 1e6)) if p.is_file() else "(folder)"))
print("ready_to_deploy flag:", dep["ready_to_deploy"], "| trial run:", dep["trial_run"], "| test evaluated:", dep["test_evaluated"])

## 25 - Final Research / Experiment Summary
**What it does:** prints one summary of the whole experiment and writes it to `06_reports/experiment_summary.md` for your documentation.
**Check:** the honest reading of the numbers - the limitation statement, the dataset caveats (duplicates, label conflicts, domain gap to face images) and whether the test set was used exactly once.

In [ ]:
def line(k, v): return "- **%s:** %s" % (k, v)
S = ["# Fitzpatrick model - experiment summary (%s)" % EXPERIMENT_NAME, "",
     line("created", datetime.datetime.now().isoformat(timespec="seconds")),
     line("classes (your folder names, Fitzpatrick order)", ", ".join(CLASS_NAMES)),
     line("images indexed / used", "%d / %d" % (len(man), int(man.use_in_experiment.sum()))),
     line("split (images)", "train %d | validation %d | test %d (split by duplicate group; one image per group in val/test)" % (len(train_df), len(val_df), len(test_df))),
     line("exact duplicates found", int(len(ok_idx) - len(reps))), line("near-duplicate groups with >1 image", int((dgrp.n > 1).sum())),
     line("cross-class (label-conflict) groups, excluded", int(len(cross))), line("flagged files for manual review", int((aud["flags"] != "").sum())),
     line("model", "%s (ImageNet) + GAP + Dropout + Dense(%d); Stage %s checkpoint chosen on validation loss" % (BACKBONE, NC, BEST_STAGE)),
     "", "## Validation", "```\n" + pd.Series(val_res).drop("n").round(4).to_string() + "\n```",
     "", "## Test (frozen)", ("Evaluated once with section 16." if TEST_RES is not None else "NOT evaluated yet (RUN_TEST_EVALUATION = False)."),
     ("```\n" + pd.Series(TEST_RES).drop("n").round(4).to_string() + "\n```" if TEST_RES is not None else ""),
     "", "## ONNX", "Agreement with Keras %.2f %%, max abs diff %.2e, mean abs diff %.2e (n = %d)." % (100 * ver["prediction_agreement"], ver["max_abs_diff"], ver["mean_abs_diff"], ver["n_images"]),
     "", "## Limitations", LIMITATION_TEXT,
     "Training images are skin photographs from a public dataset (varied body sites); webcam face captures are a different domain and must be checked separately.",
     "Near-duplicates are found heuristically; labels are those of the source folders and were not changed (flagged images are listed for manual review).",
     "This model is separate from the four-class disease classifier; the two outputs are combined only in the WPF UI, database and reports."]
open(D["reports"] / "experiment_summary.md", "w", encoding="utf-8").write("\n".join(S))
print("\n".join(S))
print("\nsaved:", D["reports"] / "experiment_summary.md")